# East vs West Atlantic — Convective Regime Comparison

**RQ3c (Geographic):** Does east-west position within the tropical Atlantic
systematically modulate convective organisation and ω profile shape?

The ORCESTRA campaign sampled two geographically distinct regions:

| Region | Base | Longitude | Period | Key environment |
|---|---|---|---|---|
| **East Atlantic** | Cape Verde / Mindelo | ~-22°W to -33°W | Aug 2024 | African dust (SAL), AEJ shear, cooler SST (cold tongue) |
| **West Atlantic** | Barbados | ~-44°W to -59°W | Sep 2024 | Clean marine air, warmer SST, moister ITCZ core |

Boundary: **-38°W** (natural data gap between the two flight legs).

Figures produced:
1. Campaign map — circle locations coloured by category and region
2. Category fraction — stacked bar comparing Top-Heavy / Bottom-Heavy / Suppressed proportions by region
3. Composite ω profiles — mean profile per region (TH and BH separately)
4. Environmental comparison — IWV and rain fraction by region
5. Top-heaviness angle distribution — violin by region

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy import stats

from scripts.organisation_metrics import build_circle_metrics, EAST_WEST_BOUNDARY_LON

COLOR_TH    = '#d94832'   # Top-Heavy  — red
COLOR_BH    = '#2b83c6'   # Bottom-Heavy — blue
COLOR_SUP   = '#6d6d6d'   # Suppressed — grey
COLOR_EAST  = '#e07b39'   # East Atlantic — warm orange
COLOR_WEST  = '#3d7ebf'   # West Atlantic — cool blue

CATEGORY_COLORS = {'Top-Heavy': COLOR_TH, 'Bottom-Heavy': COLOR_BH,
                   'Suppressed': COLOR_SUP, 'Other': '0.5'}
CATEGORY_ORDER  = ['Top-Heavy', 'Bottom-Heavy', 'Suppressed']
REGIONS         = ['East Atlantic', 'West Atlantic']
REGION_COLORS   = {'East Atlantic': COLOR_EAST, 'West Atlantic': COLOR_WEST}

## 2. Load data

In [ ]:
ds_sonde = xr.open_zarr('/g/data/k10/zr7147/ORCESTRA_dropsondes_categorized.zarr')
ds_imerg = xr.open_dataset('/g/data/k10/zr7147/ORCESTRA_IMERG_Combined_Cropped.nc')
era5     = xr.open_dataset('/g/data/k10/zr7147/ERA5/era5_single_levels.nc')

df = build_circle_metrics(ds_sonde, ds_imerg, era5)

# Exclude Missing Data circles from main analysis
df_valid = df[df['category'].isin(CATEGORY_ORDER)].copy()

print('Region counts (all circles):')
print(df.groupby('region')['circle'].count())
print()
print('Category × Region (valid circles only):')
print(pd.crosstab(df_valid['region'], df_valid['category'])[CATEGORY_ORDER])

## 3. Campaign map — circle locations by category and region

In [ ]:
fig = plt.figure(figsize=(14, 6))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

ax.set_extent([-65, -18, 0, 22], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor='0.88', zorder=0)
ax.add_feature(cfeature.OCEAN, facecolor='#ddeeff', zorder=0)
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.5)
gl = ax.gridlines(draw_labels=True, linewidth=0.4, alpha=0.4)
gl.top_labels = False
gl.right_labels = False

# East/West boundary line (axvline does not support transform on Cartopy axes)
ax.plot([EAST_WEST_BOUNDARY_LON, EAST_WEST_BOUNDARY_LON], [0, 22],
        color='0.4', lw=1.4, ls='--', transform=ccrs.PlateCarree(), zorder=2,
        label=f'E/W boundary ({EAST_WEST_BOUNDARY_LON}°W)')

# Circles coloured by category, marker shape by region
markers = {'East Atlantic': 'o', 'West Atlantic': 's'}
for region in REGIONS:
    for cat in CATEGORY_ORDER:
        sub = df_valid[(df_valid['region'] == region) & (df_valid['category'] == cat)]
        ax.scatter(
            sub['circle_lon'], sub['circle_lat'],
            color=CATEGORY_COLORS[cat],
            marker=markers[region],
            s=60, alpha=0.85, edgecolors='white', linewidths=0.5,
            transform=ccrs.PlateCarree(), zorder=3,
        )

# Legend
handles = [
    mpatches.Patch(color=COLOR_TH,  label='Top-Heavy'),
    mpatches.Patch(color=COLOR_BH,  label='Bottom-Heavy'),
    mpatches.Patch(color=COLOR_SUP, label='Suppressed'),
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='0.4',
               markersize=8, label='East Atlantic (●)'),
    plt.Line2D([0],[0], marker='s', color='w', markerfacecolor='0.4',
               markersize=8, label='West Atlantic (■)'),
    plt.Line2D([0],[0], color='0.4', lw=1.4, ls='--', label='E/W boundary'),
]
ax.legend(handles=handles, loc='lower left', fontsize=9, framealpha=0.9)

ax.set_title('ORCESTRA Dropsonde Circle Locations — Coloured by Category, Shape by Region',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/east_west_campaign_map.png', dpi=180, bbox_inches='tight')
plt.show()

## 4. Category proportions: East vs West Atlantic

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Left: stacked bar of category proportions ---
ax = axes[0]
ct = pd.crosstab(df_valid['region'], df_valid['category'])[CATEGORY_ORDER]
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

bottoms = np.zeros(len(REGIONS))
x = np.arange(len(REGIONS))
for cat in CATEGORY_ORDER:
    vals = ct_pct.loc[REGIONS, cat].values
    bars = ax.bar(x, vals, bottom=bottoms, color=CATEGORY_COLORS[cat],
                  label=cat, width=0.5, edgecolor='white', linewidth=0.8)
    for bar, v, b in zip(bars, vals, bottoms):
        if v > 5:
            ax.text(bar.get_x() + bar.get_width()/2, b + v/2,
                    f'{v:.0f}%', ha='center', va='center',
                    fontsize=10, fontweight='bold', color='white')
    bottoms += vals

# Annotate n per region
for i, region in enumerate(REGIONS):
    n = ct.loc[region].sum()
    ax.text(i, 103, f'n={n}', ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(REGIONS, fontsize=11)
ax.set_ylabel('Percentage of circles (%)', fontsize=11)
ax.set_ylim(0, 110)
ax.set_title('Category Distribution by Region', fontsize=11, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# --- Right: top-heaviness angle distribution by region ---
ax2 = axes[1]
data_by_region = [df_valid.loc[df_valid['region'] == r, 'top_heaviness_angle'].dropna().values
                  for r in REGIONS]
parts = ax2.violinplot(data_by_region, positions=[1, 2], showmedians=True)
for pc, color in zip(parts['bodies'], [COLOR_EAST, COLOR_WEST]):
    pc.set_facecolor(color)
    pc.set_alpha(0.65)
parts['cmedians'].set_colors('black')
parts['cmedians'].set_linewidths(2)

for pos, (region, data) in enumerate(zip(REGIONS, data_by_region), start=1):
    jitter = np.random.default_rng(42).uniform(-0.07, 0.07, len(data))
    # colour points by category
    sub = df_valid[df_valid['region'] == region]
    for cat in CATEGORY_ORDER:
        mask = sub['category'] == cat
        angles = sub.loc[mask, 'top_heaviness_angle'].dropna().values
        j = np.random.default_rng(pos + len(cat)).uniform(-0.07, 0.07, len(angles))
        ax2.scatter(pos + j, angles, color=CATEGORY_COLORS[cat], s=20, alpha=0.7, zorder=3,
                    edgecolors='white', linewidths=0.3)

# Mann-Whitney U test
if all(len(d) >= 3 for d in data_by_region):
    _, pval = stats.mannwhitneyu(*data_by_region, alternative='two-sided')
    sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'ns'))
    ax2.text(0.97, 0.97, f'E vs W: {sig}  (p={pval:.3f})',
             transform=ax2.transAxes, ha='right', va='top', fontsize=9,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85, edgecolor='0.7'))

ax2.axhline(0, color='0.5', lw=1, ls='--', alpha=0.6, label='TH/BH boundary')
n_by_region = [len(d) for d in data_by_region]
ax2.set_xticks([1, 2])
ax2.set_xticklabels([f'{r}\n(n={n})' for r, n in zip(REGIONS, n_by_region)], fontsize=10)
ax2.set_ylabel('Top-heaviness angle (°)', fontsize=11)
ax2.set_title('Top-heaviness Angle by Region', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.25)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig.suptitle('East vs West Atlantic — Convective Regime Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/east_west_category_distribution.png', dpi=180, bbox_inches='tight')
plt.show()

## 5. Composite ω profiles by region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 6), sharey=True)

for ax, region in zip(axes, REGIONS):
    region_circles = df_valid[df_valid['region'] == region]['circle'].values
    region_color   = REGION_COLORS[region]

    for cat, cat_color, ls in [
        ('Top-Heavy',   COLOR_TH, '-'),
        ('Bottom-Heavy', COLOR_BH, '-'),
        ('Suppressed',  COLOR_SUP, '--'),
    ]:
        cat_circles = df_valid[
            (df_valid['region'] == region) & (df_valid['category'] == cat)
        ]['circle'].values

        if len(cat_circles) == 0:
            continue

        # Stack omega and p_mean on shared altitude grid
        omega_list, p_list = [], []
        for cidx in cat_circles:
            c = ds_sonde.sel(circle=cidx)
            omega_list.append(c['omega'].values)
            p_list.append(c['p_mean'].values)

        # Use a common pressure grid (median across circles at each altitude)
        p_common = np.nanmedian(np.vstack(p_list), axis=0)
        omega_mean = np.nanmean(np.vstack(omega_list), axis=0)
        omega_std  = np.nanstd(np.vstack(omega_list), axis=0)

        valid = np.isfinite(p_common) & np.isfinite(omega_mean)
        p_v, om_v, std_v = p_common[valid], omega_mean[valid], omega_std[valid]

        ax.plot(om_v, p_v, color=cat_color, lw=2.2, ls=ls,
                label=f'{cat} (n={len(cat_circles)})')
        ax.fill_betweenx(p_v, om_v - std_v, om_v + std_v,
                         color=cat_color, alpha=0.12)

    ax.axvline(0, color='k', lw=0.9, ls='--', alpha=0.6)
    ax.set_xlabel(r'Vertical Velocity $\omega$ (Pa s$^{-1}$)', fontsize=10)
    ax.set_ylabel('Pressure (Pa)', fontsize=10)
    ax.invert_yaxis()
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8.5, framealpha=0.85)
    ax.set_title(region, fontsize=11, fontweight='bold', color=region_color)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Composite ω Profiles by Region and Category  (mean ± 1σ)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/east_west_omega_composites.png', dpi=180, bbox_inches='tight')
plt.show()

## 6. Environmental comparison: IWV and rain fraction by region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (col, ylabel, ylim) in zip(axes, [
    ('iwv_mean',     'BEACH L4 IWV  (kg m$^{-2}$)', (30, 70)),
    ('rain_fraction','Rain Area Fraction  (> 0.5 mm/hr)', (0, 0.65)),
]):
    data_by_region = [
        df_valid.loc[df_valid['region'] == r, col].dropna().values
        for r in REGIONS
    ]
    colors = [COLOR_EAST, COLOR_WEST]

    bp = ax.boxplot(
        data_by_region, patch_artist=True, widths=0.4,
        medianprops=dict(color='black', linewidth=2),
        whiskerprops=dict(linewidth=1.4),
        capprops=dict(linewidth=1.4),
        flierprops=dict(marker='o', markersize=3, alpha=0.5),
    )
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    for flier, color in zip(bp['fliers'], colors):
        flier.set_markerfacecolor(color)

    # Overlay jittered points coloured by category
    for pos, region in enumerate(REGIONS, start=1):
        sub = df_valid[df_valid['region'] == region].dropna(subset=[col])
        for cat in CATEGORY_ORDER:
            vals = sub.loc[sub['category'] == cat, col].values
            jitter = np.random.default_rng(pos).uniform(-0.12, 0.12, len(vals))
            ax.scatter(pos + jitter, vals, color=CATEGORY_COLORS[cat],
                       s=18, alpha=0.7, zorder=4, edgecolors='white', linewidths=0.3)

    # Mann-Whitney
    if all(len(d) >= 3 for d in data_by_region):
        _, pval = stats.mannwhitneyu(*data_by_region, alternative='two-sided')
        sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'ns'))
        ax.text(0.97, 0.97, f'E vs W: {sig}  (p={pval:.3f})',
                transform=ax.transAxes, ha='right', va='top', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85, edgecolor='0.7'))

    if col == 'iwv_mean':
        ax.axhline(48, color='0.4', ls='--', lw=1.1, alpha=0.7, label='~48 mm deep conv. threshold')
        ax.legend(fontsize=9)

    n_by_region = [len(d) for d in data_by_region]
    ax.set_xticks([1, 2])
    ax.set_xticklabels([f'{r}\n(n={n})' for r, n in zip(REGIONS, n_by_region)], fontsize=10)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_ylim(*ylim)
    ax.grid(axis='y', alpha=0.25)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0].set_title('Moisture Column (IWV) by Region', fontsize=11, fontweight='bold')
axes[1].set_title('Rain Fraction by Region', fontsize=11, fontweight='bold')

fig.suptitle('Environmental Conditions: East vs West Atlantic', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/east_west_environment.png', dpi=180, bbox_inches='tight')
plt.show()

## 7. Summary table

In [ ]:
summary = (
    df_valid.groupby(['region', 'category'], observed=True)
    .agg(n=('circle','count'),
         th_angle_med=('top_heaviness_angle','median'),
         iwv_med=('iwv_mean','median'),
         rain_frac_med=('rain_fraction','median'))
    .round(2)
)
print(summary.to_string())
print()

# Region-level summary (ignoring category)
region_summary = (
    df_valid.groupby('region', observed=True)
    .agg(
        n_total          = ('circle',              'count'),
        pct_top_heavy    = ('category',            lambda x: (x == 'Top-Heavy').mean() * 100),
        pct_bottom_heavy = ('category',            lambda x: (x == 'Bottom-Heavy').mean() * 100),
        pct_suppressed   = ('category',            lambda x: (x == 'Suppressed').mean() * 100),
        th_angle_median  = ('top_heaviness_angle', 'median'),
        iwv_median       = ('iwv_mean',            'median'),
        rain_frac_median = ('rain_fraction',       'median'),
    )
    .round(1)
)
region_summary.columns = ['N', '% Top-Heavy', '% Bottom-Heavy', '% Suppressed',
                          'TH angle median', 'IWV median', 'Rain frac median']
print(region_summary.to_string())